# Cleaning Of Customers Data

## Create a dataframe with proper data types

In [0]:
customers_raw_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load("/Volumes/datasets/raw/raw_volume/customers_data_csv/")

In [0]:
customers_raw_df.printSchema()


## Schema Implement

In [0]:
customer_schema = "member_id string, emp_title string, emp_length string,home_ownership string, annual_inc float, addr_state string, zip_code string,country string, grade string, sub_grade string, verification_status string,tot_hi_cred_lim float, application_type string, annual_inc_joint float,verification_status_joint string"

In [0]:
customers_raw_df_new = spark.read.format("csv").option("header", True).schema(customer_schema).load("/Volumes/datasets/raw/raw_volume/customers_data_csv/")

In [0]:
customers_raw_df_new.printSchema()

### Rename the columns

In [0]:
customers_df_renamed = customers_raw_df_new.withColumnRenamed("annual_inc","annual_income")\
                                       .withColumnRenamed("addr_State", "address_state")\
                                       .withColumnRenamed("Zip_code","Address_zipcode")\
                                       .withColumnRenamed("country","address_country")\
                                       .withColumnRenamed("tot_hi_cred_lim", "total_high_credit_limit")\
                                       .withColumnRenamed("annual_inc_joint", "join_annual_income")
                                    

In [0]:
customers_df_renamed.display()

In [0]:
customers_df_renamed.groupBy("emp_length").count().display()

In [0]:
customers_df_renamed.createOrReplaceTempView("customers_new")

In [0]:
spark.sql("select * from customers_new where emp_length like 'reactors%'").display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_updated = customers_df_renamed.withColumn(
    "emp_length",
    when(col("emp_length") == 'reactors"', None).otherwise(col("emp_length"))
)

In [0]:
df_updated.groupBy("emp_length").count().display()

## Add new Column

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
customers_df_new_column = df_updated.withColumn("ingest_date", current_timestamp())

In [0]:
customers_df_new_column.display()

In [0]:
customers_df_new_column.count()

In [0]:
customers_df_new_column.printSchema()

## REmove Duplicates

In [0]:
customer_distinct = customers_df_new_column.distinct()

In [0]:
customer_distinct.createOrReplaceTempView("customers")

In [0]:
spark.sql("select * from customers").display()


## Income Not Null

In [0]:
spark.sql("Select count(*) from customers where annual_income is null").display()

In [0]:
customers_income_filtered = spark.sql("select * from customers where annual_income is not null")

In [0]:
customers_df_new_column.count()

In [0]:
customer_distinct.count()

In [0]:
customers_income_filtered.count()

In [0]:
customers_income_filtered.display()

In [0]:
spark.sql("select distinct(emp_length) from customers").show()

In [0]:
customers_income_filtered.groupBy("emp_length").count().display()

In [0]:
spark.sql("select emp_length, count(*) from customers group by emp_length").display()

#Regular Expression

In [0]:
customers_emp_cleaned = customers_income_filtered.withColumn("emp_length",regexp_replace(col("emp_length"), r"\D", ""))


###
withColumn("emp_length", ...)
This is used to create or replace the emp_length column in the DataFrame.

col("emp_length")
This fetches the column emp_length as a column object so it can be manipulated.

regexp_replace(...)
This is a regular expression replace function that replaces parts of a string based on a regex pattern.

"(\D)"
This is the regular expression pattern:

\D means: any character that is NOT a digit (0–9)

So (\D) matches all non-digit characters

"" (empty string)
This means we’re replacing all the non-digit characters with nothing, i.e., removing them.

In [0]:
customers_emp_cleaned.display()

In [0]:
customers_emp_cleaned.count()

In [0]:
customers_emp_cleaned.printSchema()

In [0]:
customers_emp_cleaned.printSchema()

In [0]:
customers_emp_cleaned.display()

In [0]:
from pyspark.sql.functions import col, regexp_replace, when

In [0]:
customers_emp_cleaned.groupBy("emp_length").count().display()

## Convert Emp Length to Integer

In [0]:
# customers_casted = customers_emp_cleaned.withColumn(
#     "emp_length",
#     regexp_replace(col("emp_length"), r"\D", "")  # keeps only digits
# )

# Step 2: Replace empty strings with null, then cast to int
customers_casted = customers_emp_cleaned.withColumn(
    "emp_length",
    when(col("emp_length") == "", None)
    .otherwise(col("emp_length").cast("int"))
)

In [0]:
customers_casted.select("emp_length").display()

In [0]:
customers_casted.createOrReplaceTempView("customers_final")

In [0]:
avg_emp = customers_casted.selectExpr("floor(avg(emp_length)) as avg_val").first()["avg_val"]

customers_filled = customers_casted.na.fill(avg_emp, subset=["emp_length"])


In [0]:
customers_filled.printSchema()

In [0]:
customers_filled.count()

In [0]:
customers_filled.createOrReplaceTempView("customers")

In [0]:
spark.sql("select emp_length, count(*) from customers group by emp_length").display()

In [0]:
spark.sql("select * from customers").display()

In [0]:
spark.sql("select distinct(address_state) from customers").display()

In [0]:
spark.sql("select count(address_state) from customers where length(address_state) >2").display()

In [0]:
customers_state_cleaned = customers_filled.withColumn("address_state", when(length(col("address_state"))>2, "NA")\
.otherwise(col("address_state")))

In [0]:
customers_state_cleaned.count()

In [0]:
customers_state_cleaned.display()

In [0]:
customers_state_cleaned.select("address_state").distinct().display()

In [0]:
customers_state_cleaned.groupBy("address_state").count().display()

In [0]:
customers_state_cleaned_new = customers_state_cleaned.filter(customers_state_cleaned.address_state.isNotNull())

In [0]:
customers_state_cleaned_new.display()

## Write back to disk in the parquet format

In [0]:
customers_state_cleaned_new.write\
    .format("parquet")\
    .mode("overwrite")\
    .option("path","/Volumes/lending_club_project/cleaned/cleaned_volume/parquet_folder/customers_parquet")\
    .save()

## Write back the data to the disk in the csv format

In [0]:
customers_state_cleaned_new.write\
    .option("header", True)\
    .format("csv")\
    .mode("overwrite")\
    .option("path","/Volumes/lending_club_project/cleaned/cleaned_volume/csv_folder/customers_csv")\
    .save()